[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burzynski-lab/OU_PHYS2222_ComputationalPhysics/blob/main/LectureNotes/18_ML_in_Physics.ipynb)

# Machine Learning in Physics

**PHYS 2222: Computational Physics** · University of Oklahoma · Spring 2027 · Week 15

> **Before you start:** in Colab, choose **File → Save a copy in Drive** so your changes are kept.

This last notebook steps back to look at how machine learning is actually used in physics research, with four small, runnable demos along the way: finding structure without labels, a convolutional network that looks at (simulated) telescope images, a neural network that solves a differential equation, and a few experiments that show how ML analyses go wrong. It closes with advice on using AI coding assistants responsibly and on keeping Python in your toolkit after this course.

## Learning goals

After this notebook you should be able to:

- Name several areas of physics research where ML is used, and what problem it solves in each.
- Apply PCA and k-means clustering, and explain why clusters need not match the categories you care about.
- Describe what a convolutional neural network does, and train a small one on images.
- Explain the idea of a physics-informed neural network and train one to solve a simple ODE.
- Recognize data leakage, distribution shift between simulation and data, and the need for uncertainty estimates and reproducibility.
- Use AI coding assistants critically: test their code against known limits instead of trusting it.
- Set up Python on your own computer with `uv`, and use basic `git` commands.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import constants
import torch
from torch import nn

rng = np.random.default_rng(18)
torch.manual_seed(18)
device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Where ML is used in physics: a quick tour

**Particle physics at the LHC.** The Large Hadron Collider produces tens of millions of bunch crossings per second, far too many to store. **Trigger** systems decide in microseconds which collisions to keep; some now run small neural networks directly on FPGAs (reprogrammable chips) for speed. Offline, **jet tagging** identifies the particle that started a spray of particles (a "jet"): was it a bottom quark, a gluon, a top quark, a Higgs boson? Modern taggers are graph neural networks and transformers that take the full list of particles in the jet as input, and they have substantially improved the sensitivity of many measurements.

**Gamma-ray astronomy.** Exactly the problem from the last two notebooks. Cherenkov telescopes (MAGIC, H.E.S.S., VERITAS, and the next-generation Cherenkov Telescope Array Observatory) separate gamma-ray showers from the hadron background using random forests or boosted trees on Hillas parameters, and increasingly deep networks on the raw camera images.

**Astronomy and cosmology.** Surveys image hundreds of millions of galaxies, far too many for humans to classify by eye. Convolutional networks trained on labels from citizen-science projects such as Galaxy Zoo classify galaxy shapes; other networks find gravitational lenses, classify supernovae from their light curves, and estimate galaxy distances (photometric redshifts).

**Anomaly detection.** What if new physics doesn't look like any model we've thought of? Anomaly detection trains on data to learn what "normal" looks like and flags events that don't fit. It's used to search for unexpected signals at the LHC and to identify instrumental glitches in gravitational-wave detectors (the LIGO "Gravity Spy" project).

**Fast simulation with generative models.** Detailed simulations (e.g. Geant4, tracking every particle through a detector) can take most of an experiment's computing budget. **Generative models** (GANs, normalizing flows, diffusion models, the same families that generate images) learn to produce realistic simulated detector responses thousands of times faster. ATLAS already uses generative networks in part of its fast calorimeter simulation.

**Simulation-based inference.** Many physics models are easy to *simulate* but have no closed-form likelihood (notebook 15). Neural networks trained on simulations can learn the likelihood or the posterior directly, which is used in cosmology, gravitational-wave parameter estimation, and particle physics.

**Molecules and materials.** Machine-learned interatomic potentials are trained on expensive quantum-mechanical calculations, then used to run molecular dynamics with near-quantum accuracy at a tiny fraction of the cost. AlphaFold's protein-structure prediction was recognized with the 2024 Nobel Prize in Chemistry. (The 2024 Nobel Prize in Physics went to Hopfield and Hinton for foundational work on neural networks, which grew out of statistical physics.)

**Physics-informed neural networks (PINNs).** Networks trained to satisfy a differential equation, not just to fit data. You'll build one in Section 4.

A common thread: ML rarely replaces physics. It handles a step that is too high-dimensional, too slow, or too messy for traditional methods, inside an analysis that is still built on physical understanding and careful validation.

## 2. Unsupervised learning: finding structure without labels

Everything so far was **supervised**: we had labels. Often you don't. **Unsupervised learning** looks for structure in the features alone.

### k-means clustering

**k-means** splits the data into $k$ groups ("clusters"). It alternates two steps:

1. Assign each point to the nearest of $k$ cluster centers.
2. Move each center to the average position of the points assigned to it.

and repeats until nothing changes. You must choose $k$. Here it is on three clouds of **simulated** 2D points (think of any two measured quantities; the units are arbitrary).

In [ ]:
from sklearn.cluster import KMeans

centers_true = np.array([[0, 0], [4, 1], [1, 4]])
pts = np.vstack([c + rng.normal(0, 0.8, size=(150, 2)) for c in centers_true])

km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(pts)

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(pts[:, 0], pts[:, 1], c=km.labels_, s=10, cmap="viridis")
ax.plot(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1], "r*", ms=15, label="cluster centers")
ax.set_xlabel("feature 1 (arb. units)")
ax.set_ylabel("feature 2 (arb. units)")
ax.legend()
plt.show()

`np.vstack` stacks arrays on top of each other (vertically). `km.labels_` is the cluster number assigned to each point, and `c=km.labels_` colors the points by it. `n_init=10` runs the algorithm from 10 random starts and keeps the best, because k-means can get stuck in a poor solution.

### PCA: principal component analysis

The MAGIC data have 10 features, which you can't plot all at once. **Principal component analysis (PCA)** finds new axes, called **principal components**, which are combinations of the original features, ordered so that the first points along the direction in which the data vary the most, the second along the direction of most remaining variation (perpendicular to the first), and so on. Plotting the data along the first two components gives the most "spread-out" 2D view. Since PCA cares about variation, standardize the features first.

We **do not** give PCA the labels; we only use them afterward to color the plot.

In [ ]:
import os
DATA_URL = "https://raw.githubusercontent.com/burzynski-lab/OU_PHYS2222_ComputationalPhysics/main/Data/"

def data_path(filename):
    """Return a local path to a course data file if it exists, otherwise its GitHub URL."""
    for folder in ["../Data", "Data", "."]:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    return DATA_URL + filename

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

columns = ["fLength", "fWidth", "fSize", "fConc", "fConc1", "fAsym",
           "fM3Long", "fM3Trans", "fAlpha", "fDist", "class"]
magic = pd.read_csv(data_path("magic04.data"), header=None, names=columns)
features = columns[:-1]
X = magic[features].to_numpy()
y = (magic["class"] == "g").astype(int).to_numpy()

Z = StandardScaler().fit_transform(X)
pca = PCA().fit(Z)
print("fraction of variance per component:", pca.explained_variance_ratio_.round(3))

The first component carries about 42% of the total variation, and the first three together about 68%. The 10 features are strongly correlated (e.g. `fLength`, `fWidth`, and `fSize` all grow with the shower's energy), so the data effectively live in fewer than 10 dimensions.

In [ ]:
proj = pca.transform(Z)            # coordinates along the principal components
sub = rng.choice(len(y), size=4000, replace=False)   # plot a random subset to avoid clutter

fig, ax = plt.subplots(figsize=(6.5, 5))
for label, name in [(0, "hadron"), (1, "gamma")]:
    s = sub[y[sub] == label]
    ax.scatter(proj[s, 0], proj[s, 1], s=3, alpha=0.4, label=name)
ax.set_xlabel("principal component 1")
ax.set_ylabel("principal component 2")
ax.set_xlim(-5, 10)
ax.set_ylim(-6, 6)
ax.legend(markerscale=4)
plt.show()

Some hadrons stand out (the spray toward negative component 2), but most of them sit right on top of the gammas. The directions of *largest variation* are not necessarily the directions that best *separate* gammas from hadrons; much of the variation is just "big showers vs small showers" (energy). Now try k-means with $k = 2$ on the standardized features and compare the clusters to the true labels. `pd.crosstab` counts how often each combination occurs.

In [ ]:
km2 = KMeans(n_clusters=2, n_init=10, random_state=0).fit(Z)
pd.crosstab(km2.labels_, magic["class"], rownames=["cluster"], colnames=["true class"])

Both clusters are roughly two-thirds gamma, about the same as the whole data set. k-means found *some* split of the data (probably mostly by shower size), but not the gamma/hadron split. This is the central caution about unsupervised learning: **an algorithm will always find clusters, but they are not guaranteed to correspond to the physics you care about.** Unsupervised methods are excellent for exploring data, compressing it, and spotting outliers; to separate specific classes, use labels if you have them.

### Your turn 1

Run k-means on the simulated three-cloud data `pts` with $k$ = 2, 3, 4, and 6, and plot the results side by side (`plt.subplots(1, 4)`). The attribute `km.inertia_` is the total squared distance of points to their cluster centers. Plot it vs $k$ for $k = 1, \dots, 8$. Can you see from that plot that 3 is the "natural" number of clusters? (Look for an "elbow".)

In [ ]:
# Your code here

## 3. Convolutional networks on (simulated) telescope images

The Hillas parameters compress each telescope image into 10 numbers. A **convolutional neural network (CNN)** can instead learn directly from the pixels. CNNs revolutionized image recognition and are widely used on detector images in physics.

### Simulated camera images

We don't have real MAGIC images, so we **simulate** simplified ones on a $20\times 20$ pixel camera:

- **Gamma-like**: a narrow elliptical blob whose long axis points (within about 10°) toward the camera center, as for showers from a source at the center of the field of view.
- **Hadron-like**: a wider blob pointing in a random direction, sometimes with extra sub-clumps of light from sub-showers.

Each image gets a random total brightness and **Poisson noise** (photon counting statistics, as in notebook 12), plus a little background light from the night sky. This is a cartoon, not a real detector simulation, but it captures the main physics of the image shapes.

The code uses `np.mgrid`, which builds 2D arrays of pixel coordinates (like `np.meshgrid`), and rotates coordinates by an angle $\varphi$ with $u = x\cos\varphi + y\sin\varphi$, $v = -x\sin\varphi + y\cos\varphi$, so that $u$ runs along the ellipse and $v$ across it.

In [ ]:
NPIX = 20
py, px = np.mgrid[0:NPIX, 0:NPIX] - (NPIX - 1) / 2     # pixel coordinates, camera center at (0, 0)

def blob(cx, cy, sigma_long, sigma_wide, phi):
    """Normalized 2D Gaussian ellipse centered at (cx, cy), long axis at angle phi (rad)."""
    dx, dy = px - cx, py - cy
    u = dx * np.cos(phi) + dy * np.sin(phi)
    v = -dx * np.sin(phi) + dy * np.cos(phi)
    g = np.exp(-0.5 * (u / sigma_long)**2 - 0.5 * (v / sigma_wide)**2)
    return g / g.sum()

In [ ]:
def simulate_image(is_gamma):
    r, ang = rng.uniform(2, 6), rng.uniform(0, 2 * np.pi)       # image position in the camera
    cx, cy = r * np.cos(ang), r * np.sin(ang)
    if is_gamma:
        phi = ang + rng.normal(0, np.radians(10))               # points back to the center
        img = blob(cx, cy, rng.uniform(1.2, 2.5), rng.uniform(0.5, 0.9), phi)
    else:
        phi = rng.uniform(0, np.pi)                             # random direction
        img = blob(cx, cy, rng.uniform(1.2, 3.0), rng.uniform(0.7, 1.6), phi)
        for _ in range(rng.integers(0, 3)):                     # 0, 1, or 2 sub-showers
            img = img + rng.uniform(0.1, 0.4) * blob(cx + rng.normal(0, 2.5), cy + rng.normal(0, 2.5),
                                                     rng.uniform(0.5, 1.2), rng.uniform(0.5, 1.2), 0)
        img = img / img.sum()
    n_photons = rng.uniform(150, 800)
    return rng.poisson(n_photons * img + 0.3)                   # photon counts per pixel

labels = rng.integers(0, 2, size=4000)                          # 1 = gamma, 0 = hadron
images = np.array([simulate_image(lab == 1) for lab in labels])
print(images.shape)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(11, 4.6))
for row, lab, name in [(0, 1, "gamma"), (1, 0, "hadron")]:
    idx = np.flatnonzero(labels == lab)[:5]
    for ax, i in zip(axes[row], idx):
        ax.imshow(images[i], cmap="magma", origin="lower")
        ax.plot((NPIX - 1) / 2, (NPIX - 1) / 2, "c+", ms=8)    # camera center
        ax.set_title(name, fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
plt.show()

`imshow` displays a 2D array as an image; the cyan cross marks the camera center. Some cases are obvious, others are genuinely ambiguous, especially dim images where the Poisson noise dominates.

### What a convolution does

A **convolutional layer** slides a small grid of weights (a **filter**, here $3\times3$) across the image. At each position it computes the weighted sum of the $3\times3$ pixels underneath, plus a bias, producing a new image called a **feature map**. A filter with weights like
$$\begin{pmatrix} -1 & 0 & 1 \\ -1 & 0 & 1 \\ -1 & 0 & 1\end{pmatrix}$$
responds strongly to vertical edges, for example. Two key ideas:

- **The same filter is used at every position.** A pattern is recognized wherever it appears, and the layer needs only 9 weights + 1 bias per filter, not one weight per pixel.
- **The filters are learned**, not designed by hand.

A **pooling** layer (`MaxPool2d(2)`) then shrinks each feature map by keeping the maximum of every $2\times2$ block. Stacking convolution → ReLU → pooling a few times lets later layers combine small local patterns (edges, spots) into larger ones (elongated blobs, orientation relative to the center). At the end, the feature maps are flattened into one long list and fed to ordinary `Linear` layers.

PyTorch expects image batches of shape `(N, channels, height, width)`; our images have 1 channel (a color photo has 3). We take $\ln(1 + \text{counts})$ to tame the large range of brightness.

In [ ]:
X_img = torch.tensor(np.log1p(images)[:, None], dtype=torch.float32)   # shape (4000, 1, 20, 20)
y_img = torch.tensor(labels[:, None], dtype=torch.float32)
X_img_train, y_img_train = X_img[:3000].to(device), y_img[:3000].to(device)
X_img_test, y_img_test = X_img[3000:].to(device), y_img[3000:].to(device)
print(X_img_train.shape, X_img_test.shape)

In [ ]:
cnn = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1),   # 8 filters of size 3x3 -> 8 maps of 20x20
    nn.ReLU(),
    nn.MaxPool2d(2),                             # -> 8 maps of 10x10
    nn.Conv2d(8, 16, kernel_size=3, padding=1),  # 16 filters, each looking at all 8 maps -> 16 x 10x10
    nn.ReLU(),
    nn.MaxPool2d(2),                             # -> 16 x 5x5
    nn.Flatten(),                                # -> 400 numbers
    nn.Linear(16 * 5 * 5, 32),
    nn.ReLU(),
    nn.Linear(32, 1),                            # logit for "gamma"
).to(device)
print("parameters:", sum(p.numel() for p in cnn.parameters()))

`padding=1` adds a border of zeros so the output has the same size as the input. The training loop is the same as last week; only the model changed.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score, roc_curve

loader = DataLoader(TensorDataset(X_img_train, y_img_train), batch_size=64, shuffle=True)
optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(10):
    for xb, yb in loader:
        optimizer.zero_grad()
        loss_fn(cnn(xb), yb).backward()
        optimizer.step()
    with torch.no_grad():
        p_test = torch.sigmoid(cnn(X_img_test)).cpu().numpy().ravel()
    print(f"epoch {epoch}: test AUC = {roc_auc_score(labels[3000:], p_test):.4f}")

(Printing the *test* AUC every epoch is fine for a demo, but if you used it to decide when to stop, you'd need a separate validation set, as in notebook 17.)

Let's look at the gamma-like images the network finds least gamma-like, and the hadron-like images it finds most gamma-like. Looking at mistakes is one of the most useful things you can do with any classifier.

In [ ]:
test_labels = labels[3000:]
test_images = images[3000:]
worst_gamma = np.flatnonzero(test_labels == 1)[np.argsort(p_test[test_labels == 1])[:4]]
worst_hadron = np.flatnonzero(test_labels == 0)[np.argsort(p_test[test_labels == 0])[-4:]]

fig, axes = plt.subplots(2, 4, figsize=(9, 4.8))
for row, idx, name in [(0, worst_gamma, "true gamma"), (1, worst_hadron, "true hadron")]:
    for ax, i in zip(axes[row], idx):
        ax.imshow(test_images[i], cmap="magma", origin="lower")
        ax.plot((NPIX - 1) / 2, (NPIX - 1) / 2, "c+", ms=8)
        ax.set_title(f"{name}\nP(gamma) = {p_test[i]:.2f}", fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])
fig.tight_layout()
plt.show()

Typical mistakes are dim, noisy images, or hadron showers that happen to be narrow and aligned with the center. These are hard cases for any method.

### Your turn 2

(a) Train an MLP on the same images, flattened into 400 pixel values (`nn.Flatten()` as the first layer, then e.g. `nn.Linear(400, 64)`, ReLU, `nn.Linear(64, 1)`), for 10 epochs. Compare its test AUC to the CNN. Which has more parameters?
(b) **Distribution shift preview:** simulate 1000 new gamma images whose long axis points in a *random* direction (as for gammas from a diffuse source spread across the sky instead of a point source at the center). What fraction of them does the trained CNN classify as gamma (P > 0.5), compared with the original test gammas? What does that tell you about what the network learned?

In [ ]:
# Your code here

## 4. Physics-informed neural networks

A neural network trained on data knows nothing about physics, which is why the pendulum network failed to extrapolate. A **physics-informed neural network (PINN)** puts the physics into the loss function. To solve an ODE, we don't need any data at all: we ask the network $y_\theta(t)$ (where $\theta$ stands for all its parameters) to satisfy the equation itself.

Take radioactive decay, $dy/dt = -y/\tau$ with $y(0) = 1$, measuring time in units of the mean lifetime $\tau$ so that the equation is $dy/dt = -y$, with exact solution $y = e^{-t}$. Pick a set of **collocation points** $t_1, \dots, t_M$ in the interval of interest and minimize

$$L(\theta) = \underbrace{\frac{1}{M}\sum_{j=1}^{M}\left(\frac{dy_\theta}{dt}(t_j) + y_\theta(t_j)\right)^2}_{\text{the ODE should hold}} \;+\; \underbrace{\big(y_\theta(0) - 1\big)^2}_{\text{initial condition}} .$$

The derivative $dy_\theta/dt$ of the network output with respect to its *input* comes from autograd. `torch.autograd.grad(outputs, inputs, ...)` returns the derivative without storing it in `.grad`, and `create_graph=True` keeps track of it so that the loss (which contains this derivative) can itself be differentiated with respect to the parameters during training.

In [ ]:
torch.manual_seed(0)
pinn = nn.Sequential(nn.Linear(1, 20), nn.Tanh(),
                     nn.Linear(20, 20), nn.Tanh(),
                     nn.Linear(20, 1)).to(device)

t_col = torch.linspace(0, 3, 50, device=device).reshape(-1, 1).requires_grad_(True)  # collocation points
t_zero = torch.zeros(1, 1, device=device)

def pinn_loss():
    y = pinn(t_col)
    dydt = torch.autograd.grad(y, t_col, grad_outputs=torch.ones_like(y), create_graph=True)[0]
    residual = dydt + y                        # zero if the ODE dy/dt = -y is satisfied
    return (residual**2).mean() + ((pinn(t_zero) - 1)**2).mean()

In [ ]:
optimizer = torch.optim.Adam(pinn.parameters(), lr=0.01)
pinn_hist = []
for step in range(2000):
    optimizer.zero_grad()
    loss = pinn_loss()
    loss.backward()
    optimizer.step()
    pinn_hist.append(loss.item())
print(f"final loss: {pinn_hist[-1]:.2e}")

In [ ]:
t_plot = np.linspace(0, 4, 200)
with torch.no_grad():
    y_pinn = pinn(torch.tensor(t_plot[:, None], dtype=torch.float32, device=device)).cpu().numpy().ravel()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(t_plot, np.exp(-t_plot), "k--", label="exact $e^{-t}$")
axes[0].plot(t_plot, y_pinn, label="PINN")
axes[0].axvspan(0, 3, color="gray", alpha=0.12, label="collocation range")
axes[0].set_xlabel(r"time $t$ (units of $\tau$)")
axes[0].set_ylabel("$y$ (fraction remaining)")
axes[0].legend()
axes[1].semilogy(t_plot, np.abs(y_pinn - np.exp(-t_plot)))
axes[1].set_xlabel(r"time $t$ (units of $\tau$)")
axes[1].set_ylabel("|PINN − exact|")
fig.tight_layout()
plt.show()

The network found $e^{-t}$ to about $10^{-3}$ without ever seeing a single value of the solution, only the equation. Beyond $t = 3$, where it was never asked to satisfy the ODE, the error grows: a PINN, too, only knows what it was trained on.

Is this a good way to solve $dy/dt = -y$? No! `solve_ivp` (notebook 10) is faster and more accurate. PINNs become interesting for problems where traditional solvers struggle: partial differential equations on awkward geometries, problems that combine an incomplete physical model with sparse measurements, or inverse problems where an unknown parameter in the equation is learned along with the solution.

### Your turn 3

Use a PINN to solve the harmonic oscillator $\ddot{y} = -\omega^2 y$ with $\omega = 2$, $y(0) = 1$, $\dot y(0) = 0$ on $0 \le t \le 3$. You'll need the second derivative: call `torch.autograd.grad` a second time on `dydt`, again with `create_graph=True`. The loss now needs three terms (ODE residual, $y(0) = 1$, $\dot y(0) = 0$). Compare with the exact $\cos 2t$. Expect to need more training steps than for the decay; if the result is poor, try more steps or a smaller learning rate, and describe what you observe.

In [ ]:
# Your code here

## 5. Pitfalls and best practice

ML makes it easy to get an impressive-looking number that is wrong. Here are the most common traps in physics applications, with demos.

### 5.1 Data leakage

**Leakage** means information from the test data sneaks into training, making performance look better than it is. Last week's rule "fit the scaler on training data only" is one example. A more dramatic one: selecting features using the *whole* data set.

Below, the features and labels are **pure random noise**: 100 "events", each with 5000 meaningless features, and random labels. No method can truly do better than 50% accuracy. We select the 20 features most correlated with the label, then cross-validate a classifier.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

X_noise = rng.normal(size=(100, 5000))
y_noise = rng.integers(0, 2, size=100)

# WRONG: choose features using all the data, then cross-validate
X_selected = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
wrong = cross_val_score(LogisticRegression(), X_selected, y_noise, cv=5).mean()

# RIGHT: feature selection is part of the model, redone inside each training fold
right = cross_val_score(make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression()),
                        X_noise, y_noise, cv=5).mean()
print(f"leaky cross-validation accuracy:  {wrong:.2f}")
print(f"honest cross-validation accuracy: {right:.2f}")

The leaky version "finds" structure in pure noise, because the 20 features were chosen precisely because they happen to correlate with the labels of the events that later serve as validation data. Any decision made using the data (feature selection, scaling, choosing hyperparameters, choosing *which plots to make*) must be made inside the training set only.

Leakage also happens through **duplicates** and **correlated events**: if the same simulated event, or several images from the same shower, end up in both training and test sets, the test score is inflated. Physics data sets often have such hidden correlations (same run, same simulated event, neighboring time slices), so split by those groups, not by individual rows.

### 5.2 Distribution shift: simulation vs reality

In physics, classifiers are usually trained on **simulation**, where the true labels are known, then applied to **real data**. If the simulation is slightly wrong, the classifier's performance on data differs from what you measured on simulation. Suppose the simulation underestimates the image sizes by 10% (for example, because the mirror's optical blur is modeled imperfectly). We mimic this by scaling `fLength` and `fWidth` in the test set by 1.1 and applying a classifier trained on the unmodified simulation, with a threshold chosen to keep 70% of gammas in simulation.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
gbt = HistGradientBoostingClassifier(random_state=0).fit(X_train, y_train)

# threshold that keeps 70% of the simulated test gammas
score_sim = gbt.predict_proba(X_test)[:, 1]
threshold = np.quantile(score_sim[y_test == 1], 0.30)

X_shift = X_test.copy()
for name in ["fLength", "fWidth"]:
    X_shift[:, features.index(name)] *= 1.10
score_shift = gbt.predict_proba(X_shift)[:, 1]

for label, s in [("simulation", score_sim), ("shifted 'data'", score_shift)]:
    print(f"{label:15s}: signal eff = {np.mean(s[y_test == 1] > threshold):.3f},"
          f"  background eff = {np.mean(s[y_test == 0] > threshold):.3f}")

`np.quantile(a, 0.30)` is the value below which 30% of the entries of `a` lie, so 70% of the gammas score above it.

A 10% mismodeling of two features drops the signal efficiency from 70% to about 65%. That may look small, but if you used the simulated 70% to convert a number of observed gamma events into a gamma-ray flux, your flux would be off by the same factor (about 7%), with nothing in the data to warn you. Real analyses defend against this by **validating on data** wherever possible (control samples where the answer is known, such as observations of a well-measured source like the Crab Nebula, or regions with no source), by assigning **systematic uncertainties** for the simulation's imperfections, and by preferring inputs that are well modeled.

### 5.3 Uncertainty quantification

A physics result needs an uncertainty. A neural network's prediction comes with none. Approaches include the bootstrap and toy Monte Carlo methods from notebook 15, Bayesian neural networks, and **ensembles**: train several networks that differ only in their random initialization and look at the spread of their predictions. Here is an ensemble of five networks trained on last week's pendulum problem (periods for release angles 0.1–2.0 rad).

In [ ]:
from scipy.special import ellipk

def period_exact(theta0, L=1.0):
    """Exact pendulum period in s (ellipk takes m = k^2)."""
    return 4 * np.sqrt(L / constants.g) * ellipk(np.sin(theta0 / 2)**2)

theta = rng.uniform(0.1, 2.0, size=200)
T_meas = period_exact(theta) + rng.normal(0, 0.005, size=200)
X_p = torch.tensor(theta[:, None], dtype=torch.float32, device=device)
Y_p = torch.tensor(T_meas[:, None], dtype=torch.float32, device=device)
theta_plot = np.linspace(0, 3, 200)
X_plot = torch.tensor(theta_plot[:, None], dtype=torch.float32, device=device)

ensemble_preds = []
for seed in range(5):
    torch.manual_seed(seed)
    net = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 1)).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=0.01)
    for step in range(800):
        opt.zero_grad()
        nn.functional.mse_loss(net(X_p), Y_p).backward()
        opt.step()
    with torch.no_grad():
        ensemble_preds.append(net(X_plot).cpu().numpy().ravel())
ensemble_preds = np.array(ensemble_preds)          # shape (5, 200)

In [ ]:
mean, spread = ensemble_preds.mean(axis=0), ensemble_preds.std(axis=0)

fig, ax = plt.subplots()
ax.plot(theta_plot, period_exact(theta_plot), "k-", label="exact")
for p in ensemble_preds:
    ax.plot(theta_plot, p, color="C1", lw=0.8, alpha=0.6)
ax.fill_between(theta_plot, mean - 2 * spread, mean + 2 * spread, color="C1", alpha=0.25,
                label=r"ensemble mean $\pm 2\sigma$")
ax.axvspan(0.1, 2.0, color="gray", alpha=0.12, label="training range")
ax.set_ylim(1.8, 5)
ax.set_xlabel(r"release angle $\theta_0$ (rad)")
ax.set_ylabel("period $T$ (s)")
ax.legend(loc="upper left")
plt.show()

`fill_between(x, lower, upper)` shades the region between two curves. Inside the training range the five networks agree closely, and they are right. Outside it, they spread apart a bit, but strikingly they mostly **agree with each other while all being wrong**: they all level off in the same way. Out there, the spread between the networks is far smaller than their actual error.

So an ensemble captures one kind of uncertainty (the randomness of training), but not the error made by extrapolating beyond the data, where every network shares the same lack of information. Agreement between models is not proof of correctness. In physics, the defense is to know your training range and to check predictions against independent information (data, limits, physical laws).

### 5.4 Reproducibility

Someone else (including you in six months) should be able to rerun your analysis and get the same answer:

- Set random seeds (`np.random.default_rng(seed)`, `torch.manual_seed(seed)`, `random_state=` in scikit-learn).
- Record the versions of the libraries you used, as below. Results can change between versions.
- Keep your code under version control (Section 7) and make the notebook run top to bottom (*Restart and run all*).
- Save trained models (`torch.save(model.state_dict(), "model.pt")`) together with the exact data split.

In [ ]:
import sklearn, scipy, sys
print("Python", sys.version.split()[0])
for lib in [np, scipy, pd, sklearn, torch]:
    print(f"{lib.__name__:8s} {lib.__version__}")

### 5.5 Interpretability

A boosted-tree or neural-network classifier is a "black box" with thousands of parameters. In physics you want to know *why* it works, to trust it and to find out whether it exploits a real physical effect or a simulation artifact. Useful habits:

- Compare against a simple, understandable baseline (a physics-motivated cut, logistic regression).
- Check which inputs matter (permutation importance, notebook 16) and whether that makes physical sense.
- Look at the events it gets wrong, as in Section 3.
- Plot the classifier output for events in control regions where you know what to expect.
- Prefer physically meaningful, well-modeled input features where possible.

### Your turn 4

Repeat the distribution-shift study of Section 5.2, but instead shift `fSize` by $+0.04$ (that's $\log_{10}$ of the total light, so $+0.04$ means about 10% more light, as if the simulated mirrors were slightly too dim). Then shift `fAlpha` by $+3°$ (a small pointing error). Which of the three mismodelings (size, light, pointing) changes the signal efficiency most? Does it agree with what you'd guess from the permutation importances in notebook 16?

In [ ]:
# Your code here

## 6. Using AI coding assistants responsibly

Large language models (LLMs) such as ChatGPT, Claude, Gemini, and Copilot write code quickly, and they are now part of how most programmers work. The course policy allows their use, **with disclosure**, after Unit 1 (see the syllabus for details). You will also be asked about your code during your final presentation, so you need to understand everything you submit.

**What they're good at:**

- Explaining an error message or an unfamiliar function.
- Producing boilerplate: plot formatting, file loading, argument parsing.
- Translating an idea you understand into syntax you've forgotten.
- Suggesting how to restructure or vectorize code.

**How they fail:**

- **Confident, plausible, wrong code.** The code runs, the output looks reasonable, and the physics is wrong.
- **Hallucinated functions and arguments** that don't exist, or exist in a different library or version.
- **Convention mix-ups**: degrees vs radians, $\hbar$ vs $h$, Gaussian vs SI units, modulus vs parameter in elliptic integrals.
- **Silently changing your problem** into a more common, easier one.
- **Outdated APIs** from older training data.

Here is an example of the convention trap, the same one from last week. A very natural-looking pendulum function passes the modulus $k = \sin(\theta_0/2)$ to `ellipk`, which expects the parameter $m = k^2$:

In [ ]:
L, g = 1.0, constants.g
T0 = 2 * np.pi * np.sqrt(L / g)

def period_plausible(theta0):
    k = np.sin(theta0 / 2)
    return 4 * np.sqrt(L / g) * ellipk(k)        # looks right, but ellipk expects m = k**2

def period_correct(theta0):
    return 4 * np.sqrt(L / g) * ellipk(np.sin(theta0 / 2)**2)

print("Test 1: small-angle limit, T/T0 at theta0 = 1e-4 rad")
print("   plausible:", period_plausible(1e-4) / T0, "  correct:", period_correct(1e-4) / T0)

Both pass the obvious test, $T \to T_0$ as $\theta_0 \to 0$! A sharper test uses the next term of the known series expansion, $T \approx T_0\left(1 + \theta_0^2/16\right)$ for small $\theta_0$:

In [ ]:
th = 0.1
print("Test 2: T/T0 at theta0 = 0.1 rad, compared with 1 + theta0^2/16 =", 1 + th**2 / 16)
print("   plausible:", period_plausible(th) / T0)
print("   correct:  ", period_correct(th) / T0)

The wrong version is off by about 1%, small enough to miss by eye, large enough to ruin a measurement. The lesson applies to *all* code, whether it was written by an AI, a classmate, or you:

1. **Understand every line** before you use it. If you can't explain it, you can't debug it, and you can't defend it in your presentation.
2. **Test against known results**: analytic limits, conservation laws, symmetries, special cases, dimensional analysis, convergence as the step size shrinks.
3. **Read the documentation** for every library function (`help(ellipk)` would have revealed the convention).
4. **Ask for small pieces**, not whole projects, and check each one.
5. **Disclose** what you used and how.

### Your turn 5

The function below was "written by an assistant" to compute the relativistic kinetic energy of a particle. It runs without errors. Without looking for the bug first, design at least two tests based on physics you know (for example the low-speed limit $K \approx \tfrac12 mv^2$, and a case you can compute by hand such as $\gamma = 2$ at $v = (\sqrt3/2)c$), run them on an electron, and use the results to locate the bug.

In [ ]:
def kinetic_energy(m, v):
    """Relativistic kinetic energy in J, for mass m in kg and speed v in m/s."""
    gamma = 1 / np.sqrt(1 - v**2 / constants.c)
    return (gamma - 1) * m * constants.c**2

In [ ]:
# Your code here

## 7. Keep using Python after this class

Colab is convenient, but for research and for longer projects you'll want Python on your own computer. **None of the commands in this section should be run in this notebook**; they go in a terminal (Terminal on macOS, PowerShell on Windows).

### Installing Python with `uv`

[`uv`](https://docs.astral.sh/uv/) is a fast, modern tool that installs Python and manages packages for each project. Install it with

```bash
# macOS / Linux
curl -LsSf https://astral.sh/uv/install.sh | sh

# Windows (PowerShell)
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

Then create a project with its own isolated environment (a folder `.venv` holding exactly the packages this project needs):

```bash
uv init my-physics-project        # create a project folder
cd my-physics-project
uv add numpy scipy matplotlib pandas scikit-learn jupyterlab   # install packages
uv run jupyter lab                # start JupyterLab in your browser
uv run python analysis.py         # or run a script
```

Each project records its packages in `pyproject.toml` (and exact versions in `uv.lock`), which takes care of the "record your versions" advice from Section 5.4.

### Editors

- **JupyterLab** (`uv run jupyter lab`): the same notebooks you know, running locally.
- **VS Code** (free): an editor for both notebooks and `.py` scripts. Install the *Python* and *Jupyter* extensions, open your project folder, and select the `.venv` environment as the kernel/interpreter.
- As projects grow, move reusable functions out of notebooks into `.py` files and `import` them.

### Version control with git and GitHub

**git** keeps the full history of your project, so you can see what changed, undo mistakes, and collaborate. **GitHub** hosts git repositories online (this course's notebooks live there). The everyday commands:

```bash
git config --global user.name "Your Name"         # once per computer
git config --global user.email "you@example.com"

git init                       # start tracking the current folder
git status                     # what has changed?
git add analysis.py            # stage a file for the next snapshot
git commit -m "Fit decay data with likelihood"     # save a snapshot with a message
git log --oneline              # list snapshots
git diff                       # show unstaged changes line by line

git clone <url>                # download a copy of a repository
git remote add origin <url>    # connect a local repository to one on GitHub
git push                       # upload your commits
git pull                       # download others' commits
```

Add a `.gitignore` file listing things git should not track (large data files, `.venv/`, `__pycache__/`). Students can get free GitHub extras, including GitHub Codespaces (a full VS Code environment in your browser), through the GitHub Student Developer Pack.

### Where to learn more

- M. Newman, *Computational Physics* (chapters freely available from the author's website).
- J. VanderPlas, *Python Data Science Handbook*, free online.
- Scientific Python Lectures: https://lectures.scientific-python.org
- The Software Carpentry lessons (Python, the shell, git): https://software-carpentry.org/lessons/
- MIT's *The Missing Semester of Your CS Education* (shell, git, editors): https://missing.csail.mit.edu
- The scikit-learn and PyTorch tutorials, and P. Mehta et al., "A high-bias, low-variance introduction to Machine Learning for physicists", https://arxiv.org/abs/1803.08823
- If you are heading toward particle physics: the HEP Software Foundation training center, https://hepsoftwarefoundation.org/training/curriculum.html

The best way to keep your skills: use Python for your next lab report, research project, or homework calculation, even when a calculator or spreadsheet would do.

## Summary

- ML in physics: triggers and jet tagging at the LHC, gamma/hadron separation in Cherenkov telescopes, galaxy classification, anomaly detection, fast simulation with generative models, simulation-based inference, ML interatomic potentials, and physics-informed networks. ML usually handles one step inside an analysis built on physics.
- **Unsupervised learning**: `KMeans(n_clusters, n_init, random_state)` and `PCA().fit(Z)` with `explained_variance_ratio_` and `transform`. Clusters need not match the physics categories.
- **CNNs**: `nn.Conv2d`, `nn.MaxPool2d`, `nn.Flatten`; learned filters slide across an image; inputs have shape `(N, channels, height, width)`. Look at misclassified examples.
- **PINNs**: put the differential equation into the loss using `torch.autograd.grad(..., create_graph=True)`; valid only where the equation was enforced.
- **Pitfalls**: data leakage (all choices inside the training set; split by correlated groups), distribution shift between simulation and data (validate on data, assign systematics), uncertainty quantification (ensembles help but aren't proof), reproducibility (seeds, versions, version control), interpretability (baselines, importances, mistakes, control regions).
- **AI assistants**: useful but fallible. Understand every line, test against known limits, read the docs, disclose your use.
- **After the course**: `uv` to install Python and packages, JupyterLab or VS Code to edit, `git` + GitHub to keep history and share.

Congratulations on finishing PHYS 2222. In one semester you went from `print("Hello")` to differential equations, Monte Carlo simulations, likelihood fits, and neural networks: the same tools physicists use every day in research.